In [1]:
import torch
import os
import gradio as gr
# from ibm_watsonx_ai.foundation_models.utils.enums import ModelTypes
# from ibm_watsonx_ai import APIClient, Credentials
# from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
# from ibm_watsonx_ai.foundation_models.utils.enums import DecodingMethods
# from ibm_watsonx_ai.foundation_models.schema import TextChatParameters
# from ibm_watsonx_ai.foundation_models import ModelInference
# from langchain_ibm import WatsonxLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from transformers import pipeline  # For Speech-to-Text

In [3]:
#######------------- LLM Initialization-------------#######

# IBM Watsonx LLM credentials
# project_id = "skills-network"

# credentials = Credentials(
#                    url="https://us-south.ml.cloud.ibm.com",
#                    # api_key="<YOUR_API_KEY>"  # Normally you'd put an API key here
#                   )
# client = APIClient(credentials)
# model_id = "ibm/granite-3-3-8b-instruct"

# parameters = {
#     GenParams.DECODING_METHOD: "sample",
#     GenParams.MAX_NEW_TOKENS: 512,
#     GenParams.MIN_NEW_TOKENS: 1,
#     GenParams.TEMPERATURE: 0.5,
#     GenParams.TOP_K: 50,
#     GenParams.TOP_P: 1,
# }

# Initialize IBM Watsonx LLM
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="phi3:mini", base_url="http://127.0.0.1:11434")
print(llm.invoke("Hi"))

Hello, how can I help you today?


In [4]:
#######------------- Helper Functions-------------#######

# Function to remove non-ASCII characters
def remove_non_ascii(text):
    return ''.join(i for i in text if ord(i) < 128)



In [10]:
def product_assistant(ascii_transcript):
    system_prompt = """You are an intelligent assistant specializing in financial products;
    your task is to process transcripts of earnings calls, ensuring that all references to
     financial products and common financial terms are in the correct format. For each
     financial product or common term that is typically abbreviated as an acronym, the full term 
    should be spelled out followed by the acronym in parentheses. For example, '401k' should be
     transformed to '401(k) retirement savings plan', 'HSA' should be transformed to 'Health Savings Account (HSA)' , 'ROA' should be transformed to 'Return on Assets (ROA)', 'VaR' should be transformed to 'Value at Risk (VaR)', and 'PB' should be transformed to 'Price to Book (PB) ratio'. Similarly, transform spoken numbers representing financial products into their numeric representations, followed by the full name of the product in parentheses. For instance, 'five two nine' to '529 (Education Savings Plan)' and 'four zero one k' to '401(k) (Retirement Savings Plan)'. However, be aware that some acronyms can have different meanings based on the context (e.g., 'LTV' can stand for 'Loan to Value' or 'Lifetime Value'). You will need to discern from the context which term is being referred to  and apply the appropriate transformation. In cases where numerical figures or metrics are spelled out but do not represent specific financial products (like 'twenty three percent'), these should be left as is. Your role is to analyze and adjust financial product terminology in the text. Once you've done that, produce the adjusted transcript and a list of the words you've changed"""

    # Concatenate the system prompt and the user transcript
    prompt_input = system_prompt + "\n" + ascii_transcript

    # Create a messages object
    messages = [
        {
            "role": "user",
            "content": prompt_input
        }
    ]

    # Construct the model ID using the specified model size
    # model_id = f"meta-llama/llama-3-2-11b-vision-instruct"
    
    # Configure the parameters for model behavior
    # params = TextChatParameters(
    #     temperature=0.2,  # Controls randomness; lower values make the output more deterministic
    #     top_p=0.6              # Nucleus sampling to control output diversity
    # )
    
    # # Initialize the Llama 3.2 model inference object
    # llama32 = ModelInference(
    #     model_id=model_id,         # Specify the model ID to use (Llama 3.2)
    #     credentials=credentials,   # Authentication credentials for accessing the model
    #     project_id=project_id,     # Link to the associated project ID
    #     params=params              # Parameters that define the model's response behavior
    # )
    
    # Send the input messages to the model and retrieve its response
    # response = llama32.chat(messages=messages)
    response = llm.invoke(messages)
    
    # Extract and return the content of the model's first response choice
    # return response['choices'][0]['message']['content']
    return response


In [12]:
#######------------- Prompt Template and Chain-------------#######

# Define the prompt template
template = """
Generate meeting minutes and a list of tasks based on the provided context.

Context:
{context}

Meeting Minutes:
- Key points discussed
- Decisions made

Task List:
- Actionable items with assignees and deadlines
"""

prompt = ChatPromptTemplate.from_template(template)

# Define the chain
chain = (
    {"context": RunnablePassthrough()}  # Pass the transcript as context
    | prompt
    | llm
    | StrOutputParser()
)



In [13]:
#######------------- Speech2text and Pipeline-------------#######

# Speech-to-text pipeline
def transcript_audio(audio_file):
    pipe = pipeline("automatic-speech-recognition",model="openai/whisper-tiny.en",chunk_length_s=30)
    raw_transcript = pipe(audio_file, batch_size=8)["text"]
    ascii_transcript = remove_non_ascii(raw_transcript)

    adjusted_transcript = product_assistant(ascii_transcript)
    result = chain.invoke({"context": adjusted_transcript})

    # Write the result to a file for downloading
    output_file = "meeting_minutes_and_tasks.txt"
    with open(output_file, "w") as file:
        file.write(result)

    # Return the textual result and the file for download
    return result, output_file




In [ ]:
#######------------- Gradio Interface-------------#######

audio_input = gr.Audio(sources="upload", type="filepath", label="Upload your audio file")
output_text = gr.Textbox(label="Meeting Minutes and Tasks")
download_file = gr.File(label="Download the Generated Meeting Minutes and Tasks")

iface = gr.Interface(
    fn=transcript_audio,
    inputs=audio_input,
    outputs=[output_text, download_file],
    title="AI Meeting Assistant",
    description="Upload an audio file of a meeting. This tool will transcribe the audio, fix product-related terminology, and generate meeting minutes along with a list of tasks."
)

iface.launch(server_name="127.0.0.1", server_port=7862)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Device set to use cpu
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
